In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd


/home/dlaschenko/transformers-activation-approximation/transformer-activation-approximation/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

data = pd.read_parquet("../data/validation.parquet")

test_data = data["sentence"].tolist()
labels = data["label"].tolist()

print(f"Dataset has {len(test_data)} rows.")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6598.61it/s]


Dataset has 872 rows.
Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti


In [3]:
def evaluate_model(model, tokenizer, texts, batch_size=32):
    model.eval()

    all_logits = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():
            output = model(**tokens)

        all_logits.append(output.logits.cpu())

    return torch.cat(all_logits)

In [4]:
baseline_logits = evaluate_model(model, tokenizer, test_data)

print("Baseline: ", baseline_logits)

Baseline:  tensor([[-4.3610,  4.7005],
        [ 3.7257, -3.1509],
        [-3.8708,  4.0583],
        ...,
        [ 1.3311, -1.0308],
        [ 3.6481, -3.0162],
        [ 0.7629, -0.4923]])


In [5]:
def custom_activation(x):
    return torch.nn.functional.gelu(x)

class CustomActivation(torch.nn.Module):
    def forward(self, x):
        return custom_activation(x)

original_activations = [
    block.ffn.activation
    for block in model.distilbert.transformer.layer
]

for block in model.distilbert.transformer.layer:
    block.ffn.activation = CustomActivation()


In [6]:
modified_logits = evaluate_model(model, tokenizer, test_data)

print("Baseline:", baseline_logits)
print("Modified:", modified_logits)

baseline_classes = baseline_logits.argmax(dim=-1)
modified_classes = modified_logits.argmax(dim=-1)
print("Baseline classes:", baseline_classes)
print("Modified classes:", modified_classes)

Baseline: tensor([[-4.3610,  4.7005],
        [ 3.7257, -3.1509],
        [-3.8708,  4.0583],
        ...,
        [ 1.3311, -1.0308],
        [ 3.6481, -3.0162],
        [ 0.7629, -0.4923]])
Modified: tensor([[-4.3610,  4.7005],
        [ 3.7257, -3.1509],
        [-3.8708,  4.0583],
        ...,
        [ 1.3311, -1.0308],
        [ 3.6481, -3.0162],
        [ 0.7629, -0.4923]])
Baseline classes: tensor([1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1,
        1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,
        1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1,
        1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1,
        0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,
        0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 

In [7]:
baseline_correct = 0
modified_correct = 0

for index, label in enumerate(labels):
    if labels[index] == baseline_classes[index]:
         baseline_correct += 1

    if labels[index] == modified_classes[index]:
         modified_correct += 1

baseline_accuracy = baseline_correct/len(labels)
modified_accuracy = modified_correct/len(labels)
print(baseline_accuracy)
print(modified_accuracy)



0.9105504587155964
0.9105504587155964
